In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")  # allows saving plots to file without a display
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")

##### STEP 1: LOAD THE DATA

In [3]:
df = pd.read_csv("data/delhi_house_data.csv")
df.shape

(2000, 13)

In [4]:
df.head()

,Locality,Property_Type,BHK,Bathrooms,Balconies,Area_SqFt,Age_Years,Floor_No,Total_Floors,Furnishing,Parking_Spaces,Distance_From_Metro_KM,Price_INR
0,Dwarka,Builder Floor,3,3,0,1563.1,10,7,12,Semi-Furnished,1,0.04,11625000.0
1,Golf Links,Builder Floor,2,2,3,1406.7,14,13,16,Unfurnished,2,1.26,25465000.0
2,Defence Colony,Independent House,1,1,1,1759.9,6,11,19,Unfurnished,2,0.60,29829000.0
3,Rohini,Builder Floor,3,2,2,986.6,33,9,13,Semi-Furnished,2,0.19,5678000.0
4,Rohini,Builder Floor,3,3,2,1479.0,28,14,15,Unfurnished,1,1.00,9832000.0


In [5]:
df.dtypes

Locality                   object
Property_Type              object
BHK                         int64
Bathrooms                   int64
Balconies                   int64
Area_SqFt                 float64
Age_Years                   int64
Floor_No                    int64
Total_Floors                int64
Furnishing                 object
Parking_Spaces              int64
Distance_From_Metro_KM    float64
Price_INR                 float64
dtype: object

#### STEP 2: EXPLORE THE DATA (EDA - EXPLORATARY DATA ANALYSIS)

In [6]:
df.isnull().sum()

Locality                  0
Property_Type             0
BHK                       0
Bathrooms                 0
Balconies                 0
Area_SqFt                 0
Age_Years                 0
Floor_No                  0
Total_Floors              0
Furnishing                0
Parking_Spaces            0
Distance_From_Metro_KM    0
Price_INR                 0
dtype: int64

In [7]:
df.describe()

,BHK,Bathrooms,Balconies,Area_SqFt,Age_Years,Floor_No,Total_Floors,Parking_Spaces,Distance_From_Metro_KM,Price_INR
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.00000,2000.000000,2000.000000,2000.00000,2000.000000,2.000000e+03
mean,2.688500,2.427000,1.287000,1411.896100,19.55350,7.127500,12.166000,0.90350,2.046785,1.348508e+07
std,0.988411,0.783567,0.919268,607.590937,11.55244,4.335856,5.093173,0.69601,2.062578,9.979075e+06
min,1.000000,1.000000,0.000000,300.000000,0.00000,0.000000,1.000000,0.00000,0.000000,1.048000e+06
25%,2.000000,2.000000,1.000000,1031.950000,9.00000,3.000000,8.000000,0.00000,0.590000,7.171500e+06
50%,3.000000,2.000000,1.000000,1282.150000,20.00000,7.000000,12.000000,1.00000,1.360000,1.062200e+07
75%,3.000000,3.000000,2.000000,1624.125000,30.00000,11.000000,16.000000,1.00000,2.812500,1.684500e+07
max,5.000000,5.000000,3.000000,4653.900000,39.00000,14.000000,23.000000,2.00000,15.000000,9.896800e+07


In [8]:
df["Price_Lakh"] = df["Price_INR"] / 100000

In [9]:
df.describe()

,BHK,Bathrooms,Balconies,Area_SqFt,Age_Years,Floor_No,Total_Floors,Parking_Spaces,Distance_From_Metro_KM,Price_INR,Price_Lakh
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.00000,2000.000000,2000.000000,2000.00000,2000.000000,2.000000e+03,2000.000000
mean,2.688500,2.427000,1.287000,1411.896100,19.55350,7.127500,12.166000,0.90350,2.046785,1.348508e+07,134.850845
std,0.988411,0.783567,0.919268,607.590937,11.55244,4.335856,5.093173,0.69601,2.062578,9.979075e+06,99.790746
min,1.000000,1.000000,0.000000,300.000000,0.00000,0.000000,1.000000,0.00000,0.000000,1.048000e+06,10.480000
25%,2.000000,2.000000,1.000000,1031.950000,9.00000,3.000000,8.000000,0.00000,0.590000,7.171500e+06,71.715000
50%,3.000000,2.000000,1.000000,1282.150000,20.00000,7.000000,12.000000,1.00000,1.360000,1.062200e+07,106.220000
75%,3.000000,3.000000,2.000000,1624.125000,30.00000,11.000000,16.000000,1.00000,2.812500,1.684500e+07,168.450000
max,5.000000,5.000000,3.000000,4653.900000,39.00000,14.000000,23.000000,2.00000,15.000000,9.896800e+07,989.680000


In [10]:
# --- Plot 1: Distribution of house prices ---
plt.figure(figsize=(8, 5))
sns.histplot(df["Price_Lakh"], bins=40, kde=True, color="teal")
plt.title("Distribution of House Prices in Delhi (Lakhs INR)")
plt.xlabel("Price (Lakh INR)")
plt.ylabel("Number of houses")
plt.tight_layout()
plt.savefig("outputs/price_distribution.png", dpi=120)
plt.close()
print("Saved: outputs/price_distribution.png")

Saved: outputs/price_distribution.png


In [12]:
# --- Plot 2: Price vs Area (scatter) ---
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Area_SqFt", y="Price_Lakh", hue="BHK", palette="viridis", alpha=0.6)
plt.title("House Price vs Area (colored by BHK)")
plt.xlabel("Area (Sq. Ft.)")
plt.ylabel("Price (Lakh INR)")
plt.tight_layout()
plt.savefig("outputs/price_vs_area.png", dpi=120)
plt.close()
print("Saved: outputs/price_vs_area.png")

Saved: outputs/price_vs_area.png


In [ ]:
# --- Plot 3: Average price by locality ---
plt.figure(figsize=(9, 6))
avg_price_by_locality = (df.groupby("Locality")["Price_Lakh"].mean().sort_values(ascending=False))
sns.barplot(x=avg_price_by_locality.values, y=avg_price_by_locality.index, palette="mako")
plt.title("Average House Price by Locality")
plt.xlabel("Average Price (Lakh INR)")
plt.tight_layout()
plt.savefig("outputs/avg_price_by_locality.png", dpi=120)
plt.close()
print("Saved: outputs/avg_price_by_locality.png")

C:\Users\akash\AppData\Local\Temp\ipykernel_20304\3636234008.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=avg_price_by_locality.values, y=avg_price_by_locality.index, palette="mako")


Saved: outputs/avg_price_by_locality.png


In [14]:
# --- Plot 4: Correlation heatmap (numeric features only) ---
plt.figure(figsize=(8, 6))
numeric_cols = df.select_dtypes(include=[np.number]).drop(columns=["Price_INR"])
corr = numeric_cols.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Between Numeric Features")
plt.tight_layout()
plt.savefig("outputs/correlation_heatmap.png", dpi=120)
plt.close()
print("Saved: outputs/correlation_heatmap.png")

Saved: outputs/correlation_heatmap.png


#### STEP 3: PREPARE THE DATA FOR MACHINE LEARNING

In [15]:
TARGET = "Price_INR"
FEATURES = [
    "Locality", "Property_Type", "BHK", "Bathrooms", "Balconies",
    "Area_SqFt", "Age_Years", "Floor_No", "Total_Floors",
    "Furnishing", "Parking_Spaces", "Distance_From_Metro_KM",
]

X = df[FEATURES]
y = df[TARGET]

categorical_features = ["Locality", "Property_Type", "Furnishing"]
numeric_features = [c for c in FEATURES if c not in categorical_features]

print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")

Numeric features: ['BHK', 'Bathrooms', 'Balconies', 'Area_SqFt', 'Age_Years', 'Floor_No', 'Total_Floors', 'Parking_Spaces', 'Distance_From_Metro_KM']
Categorical features: ['Locality', 'Property_Type', 'Furnishing']


In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough",  # keep numeric columns as they are
)

#### STEP 4: TRAIN / TEST SPLIT

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training set: {X_train.shape[0]} houses")
print(f"Test set:     {X_test.shape[0]} houses")

Training set: 1600 houses
Test set:     400 houses


#### STEP 5 & 6: TRAIN MODELS

In [19]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42, max_depth=12),
}

trained_pipelines = {}
results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline

    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({"Model": name, "MAE (INR)": mae, "RMSE (INR)": rmse, "R2 Score": r2})

    print(f"\n--- {name} ---")
    print(f"MAE  (avg error in INR):        {mae:,.0f}")
    print(f"RMSE (penalizes big errors):    {rmse:,.0f}")
    print(f"R2 Score (0-1, higher=better):  {r2:.3f}")


--- Linear Regression ---
MAE  (avg error in INR):        2,011,176
RMSE (penalizes big errors):    2,781,301
R2 Score (0-1, higher=better):  0.892

--- Random Forest ---
MAE  (avg error in INR):        1,946,604
RMSE (penalizes big errors):    2,647,472
R2 Score (0-1, higher=better):  0.902


#### STEP 7: COMPARE MODELS

In [20]:
results_df = pd.DataFrame(results).sort_values("R2 Score", ascending=False)
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]["Model"]
best_pipeline = trained_pipelines[best_model_name]
print(f"\nBest model: {best_model_name}")

            Model    MAE (INR)   RMSE (INR)  R2 Score
    Random Forest 1.946604e+06 2.647472e+06  0.902327
Linear Regression 2.011176e+06 2.781301e+06  0.892203

Best model: Random Forest


In [21]:
# Plot: Predicted vs Actual prices for the best model
y_pred_best = best_pipeline.predict(X_test)
plt.figure(figsize=(7, 7))
plt.scatter(y_test / 100000, y_pred_best / 100000, alpha=0.5, color="darkorange")
max_val = max(y_test.max(), y_pred_best.max()) / 100000
plt.plot([0, max_val], [0, max_val], "--", color="gray")  # perfect-prediction line
plt.xlabel("Actual Price (Lakh INR)")
plt.ylabel("Predicted Price (Lakh INR)")
plt.title(f"Predicted vs Actual Prices ({best_model_name})")
plt.tight_layout()
plt.savefig("outputs/predicted_vs_actual.png", dpi=120)
plt.close()
print("Saved: outputs/predicted_vs_actual.png")

Saved: outputs/predicted_vs_actual.png


#### STEP 8: FEATURE IMPORTANCE (only meaningful for Random Forest)

In [22]:
rf_pipeline = trained_pipelines["Random Forest"]
ohe = rf_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
cat_feature_names = list(ohe.get_feature_names_out(categorical_features))
all_feature_names = cat_feature_names + numeric_features

importances = rf_pipeline.named_steps["model"].feature_importances_
importance_df = pd.DataFrame({
    "Feature": all_feature_names,
    "Importance": importances,
}).sort_values("Importance", ascending=False).head(15)

print(importance_df.to_string(index=False))

                 Feature  Importance
               Area_SqFt    0.531367
     Locality_Golf Links    0.118145
   Locality_Vasant Vihar    0.117186
 Locality_Defence Colony    0.060356
Locality_Greater Kailash    0.032450
      Locality_Hauz Khas    0.022821
          Locality_Saket    0.015822
               Age_Years    0.015738
      Locality_Najafgarh    0.013039
                     BHK    0.009714
  Distance_From_Metro_KM    0.008831
                Floor_No    0.006753
     Locality_Model Town    0.006690
            Total_Floors    0.006286
               Bathrooms    0.004826


In [23]:
plt.figure(figsize=(8, 6))
sns.barplot(data=importance_df, x="Importance", y="Feature", palette="crest")
plt.title("Top 15 Most Important Features (Random Forest)")
plt.tight_layout()
plt.savefig("outputs/feature_importance.png", dpi=120)
plt.close()
print("Saved: outputs/feature_importance.png")

Saved: outputs/feature_importance.png


C:\Users\akash\AppData\Local\Temp\ipykernel_20304\547036410.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=importance_df, x="Importance", y="Feature", palette="crest")


In [27]:
# Save trained model
joblib.dump(best_pipeline, "outputs/best_model.joblib")
print(f"\nSaved trained model to outputs/best_model.joblib")


Saved trained model to outputs/best_model.joblib


#### STEP 9: TRY A PREDICTION ON A NEW, MADE-UP HOUSE

In [25]:
new_house = pd.DataFrame([{
    "Locality": "Dwarka",
    "Property_Type": "Apartment",
    "BHK": 3,
    "Bathrooms": 2,
    "Balconies": 2,
    "Area_SqFt": 1450,
    "Age_Years": 5,
    "Floor_No": 4,
    "Total_Floors": 10,
    "Furnishing": "Semi-Furnished",
    "Parking_Spaces": 1,
    "Distance_From_Metro_KM": 1.2,
}])

In [26]:
predicted_price = best_pipeline.predict(new_house)[0]
print("New house details:")
print(new_house.to_string(index=False))
print(f"\nPredicted price: Rs {predicted_price:,.0f}  (~ Rs {predicted_price/100000:.1f} Lakh)")

print("\nDone! Check the 'outputs/' folder for saved charts and the trained model.")

New house details:
Locality Property_Type  BHK  Bathrooms  Balconies  Area_SqFt  Age_Years  Floor_No  Total_Floors     Furnishing  Parking_Spaces  Distance_From_Metro_KM
  Dwarka     Apartment    3          2          2       1450          5         4            10 Semi-Furnished               1                     1.2

Predicted price: Rs 10,066,437  (~ Rs 100.7 Lakh)

Done! Check the 'outputs/' folder for saved charts and the trained model.
